# Notebook 02 — Preprocesamiento

En este notebook transformamos el texto crudo en datos limpios
y estructurados listos para entrenar un modelo de Machine Learning.

El proceso sigue este orden:
1. Limpieza del texto
2. Tokenización y eliminación de stopwords
3. Lematización
4. División train/test
5. Vectorización con TF-IDF

In [2]:
# Librerías de manejo de datos
import pandas as pd
import numpy as np

# Librería para expresiones regulares (limpieza de texto)
import re

# Librería para NLP básico
import nltk
from nltk.corpus   import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem     import WordNetLemmatizer

# Descargar recursos de NLTK necesarios
nltk.download('stopwords',    quiet=True)
nltk.download('punkt',        quiet=True)
nltk.download('punkt_tab',    quiet=True)
nltk.download('wordnet',      quiet=True)

# Librería para dividir datos en train y test
from sklearn.model_selection import train_test_split

# Librería para vectorización
from sklearn.feature_extraction.text import TfidfVectorizer

# Librería para guardar objetos en disco
import joblib

# Librería para guardar matrices dispersas
import scipy.sparse as sp

# Cargamos el dataset original
df = pd.read_csv('../../data/raw/youtoxic_english_1000.csv')

# Comprobamos que cargó bien
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head(3)

Filas: 1000 | Columnas: 15


,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\r\nDont you reckon them 'black lives matter' ...,True,True,False,False,True,False,False,False,False,False,False,False


# 2. Limpieza del texto

Antes de tokenizar o vectorizar, necesitamos limpiar el texto crudo.
Construimos una función que aplica todos los pasos de limpieza
en el orden correcto.

Pasos:
1. Convertir a minúsculas
2. Eliminar URLs
3. Eliminar menciones (@usuario)
4. Eliminar hashtags (#tema)
5. Eliminar emojis
6. Eliminar puntuación y caracteres especiales
7. Eliminar espacios extra

In [3]:
def clean_text(text):
    """
    Limpia un texto aplicando los siguientes pasos:
    1. Minúsculas
    2. Eliminar URLs
    3. Eliminar menciones
    4. Eliminar hashtags
    5. Eliminar emojis y caracteres no ASCII
    6. Eliminar puntuación y caracteres especiales
    7. Eliminar espacios extra
    """
    # 1. Minúsculas
    text = text.lower()

    # 2. Eliminar URLs
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+',  '', text)

    # 3. Eliminar menciones
    text = re.sub(r'@\S+', '', text)

    # 4. Eliminar hashtags
    text = re.sub(r'#\S+', '', text)

    # 5. Eliminar emojis y caracteres no ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 6. Eliminar puntuación y caracteres especiales
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 7. Eliminar espacios extra
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

In [4]:
# Probamos con un ejemplo antes de aplicarla a todo el dataset
ejemplo = "Check this out!! 😡 http://youtube.com @usuario #BlackLivesMatter"
print("Antes:", ejemplo)
print("Después:", clean_text(ejemplo))

Antes: Check this out!! 😡 http://youtube.com @usuario #BlackLivesMatter
Después: check this out


**Verificación:**
La función de limpieza funciona correctamente.

- Convierte a minúsculas ✓
- Elimina URLs ✓
- Elimina menciones ✓
- Elimina hashtags ✓
- Elimina emojis ✓
- Elimina puntuación ✓

El texto resultante contiene solo palabras en minúsculas
sin ruido. Listo para tokenizar.

# 3. Tokenización y eliminación de stopwords

Una vez limpio el texto, lo dividimos en palabras individuales
y eliminamos las palabras vacías que no aportan significado.

Pasos:
1. Tokenizar: dividir el texto en palabras
2. Eliminar stopwords: quitar palabras vacías en inglés

In [5]:
# Cargamos las stopwords en inglés
stop_words = set(stopwords.words('english'))

def tokenize_and_remove_stopwords(text):
    """
    Tokeniza el texto y elimina stopwords y palabras cortas.
    Devuelve el texto como string limpio.
    """
    # Tokenizamos
    tokens = word_tokenize(text)

    # Filtramos stopwords y palabras muy cortas
    tokens = [
        token for token in tokens
        if token not in stop_words
        and len(token) > 2
    ]

    # Devolvemos como texto
    return ' '.join(tokens)

In [6]:
# Probamos con el resultado del bloque anterior
ejemplo_limpio = clean_text(
    "Black people are getting shot by police officers every day"
)
print("Después de limpiar:", ejemplo_limpio)
print("Después de tokenizar:", tokenize_and_remove_stopwords(ejemplo_limpio))

Después de limpiar: black people are getting shot by police officers every day
Después de tokenizar: black people getting shot police officers every day


**Verificación:**
La tokenización y eliminación de stopwords funciona correctamente.

- Tokenización: el texto se divide en palabras individuales ✓
- Stopwords eliminadas: are, by, every ✓
- Palabras con significado conservadas: black, people,
  shot, police, officers ✓

Las palabras clave para detectar toxicidad se mantienen.
Listo para lematizar.

# 4. Lematización

La lematización reduce cada palabra a su forma base (lema).
Esto agrupa variantes de la misma palabra para que el modelo
las trate como una sola cosa.

Ejemplos:
- running, runs, ran  → run
- officers, officer   → officer
- getting, gets, got  → get
- people, person      → person

In [7]:
# Creamos el lematizador
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    """
    Lematiza cada palabra del texto reduciéndola a su forma base.
    Intenta primero como verbo y si no cambia, como sustantivo.
    """
    tokens = []

    for token in text.split():
        # Intentamos como verbo primero
        lemma_v = lemmatizer.lemmatize(token, pos='v')
        if lemma_v != token:
            tokens.append(lemma_v)
        else:
            tokens.append(lemmatizer.lemmatize(token))

    return ' '.join(tokens)

In [8]:
# Encadenamos los tres pasos
ejemplo = "Black people are getting shot by police officers every day"

paso1 = clean_text(ejemplo)
paso2 = tokenize_and_remove_stopwords(paso1)
paso3 = lemmatize_text(paso2)

print("Original: ", ejemplo)
print("Limpio:   ", paso1)
print("Tokenizado:", paso2)
print("Lematizado:", paso3)

Original:  Black people are getting shot by police officers every day
Limpio:    black people are getting shot by police officers every day
Tokenizado: black people getting shot police officers every day
Lematizado: black people get shoot police officer every day


**Verificación:**
La lematización funciona correctamente.

Cambios aplicados:
- getting → get (verbo a forma base) ✓
- officers → officer (plural a singular) ✓
- shot → shoot (pasado a infinitivo) ✓

Nota: "shot" se lematiza como "shoot" porque el lematizador
lo interpreta como verbo. Es una limitación del enfoque
clásico sin contexto. BERT resolvería este caso correctamente.

El texto está listo para aplicar el pipeline completo
al dataset.

# 5. Pipeline completo de preprocesamiento

Aplicamos las tres funciones encadenadas a todo el dataset:
1. clean_text → limpia el texto
2. tokenize_and_remove_stopwords → elimina palabras vacías
3. lemmatize_text → reduce palabras a su forma base

El resultado se guarda en una nueva columna: Text_clean

In [9]:
# --- Pipeline completo ---

# Nos quedamos solo con las columnas necesarias
df = df[['Text', 'IsToxic']]

# Función pipeline completa
def full_pipeline(text):
    """
    Aplica limpieza, tokenización y lematización en orden.
    """
    text = clean_text(text)
    text = tokenize_and_remove_stopwords(text)
    text = lemmatize_text(text)
    return text

# Aplicamos el pipeline a todo el dataset
print("Procesando el dataset...")
df['Text_clean'] = df['Text'].apply(full_pipeline)
print("Procesamiento completado.")

# Comparamos original vs limpio
df[['Text', 'Text_clean']].head(5)

Procesando el dataset...
Procesamiento completado.


,Text,Text_clean
0,If only people would just take a step back and...,people would take step back make case wasnt an...
1,Law enforcement is not trained to shoot to app...,law enforcement train shoot apprehend train sh...
2,\r\nDont you reckon them 'black lives matter' ...,dont reckon black live matter banner hold whit...
3,There are a very large number of people who do...,large number people like police officer call c...
4,"The Arab dude is absolutely right, he should h...",arab dude absolutely right shoot extra time sh...


**Verificación del pipeline completo:**

El preprocesamiento se aplicó correctamente a las 1000 filas.

Transformaciones observadas:
- Stopwords eliminadas correctamente ✓
- Minúsculas aplicadas ✓
- Caracteres especiales eliminados (\r\n, comillas) ✓
- Lematización aplicada: trained→train, lives→live,
  officers→officer ✓
- No hay textos vacíos ✓

La columna Text_clean está lista para dividir en
train/test y vectorizar.

# 6. División train / test

Dividimos el dataset en dos partes:
- Train (entrenamiento): el modelo aprende con estos datos
- Test (evaluación): comprobamos si el modelo aprendió bien

Esta división es fundamental para detectar overfitting.

In [10]:
# --- División train / test ---

# Separamos entrada y salida
X = df['Text_clean']
y = df['IsToxic']

# Dividimos el dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Comprobamos las dimensiones
print(f"X_train: {X_train.shape[0]} filas")
print(f"X_test:  {X_test.shape[0]} filas")
print(f"y_train: {y_train.shape[0]} filas")
print(f"y_test:  {y_test.shape[0]} filas")

X_train: 800 filas
X_test:  200 filas
y_train: 800 filas
y_test:  200 filas


In [11]:
# Verificamos la distribución de clases
print("\nDistribución en train:")
print(y_train.value_counts(normalize=True).round(3) * 100)

print("\nDistribución en test:")
print(y_test.value_counts(normalize=True).round(3) * 100)


Distribución en train:
IsToxic
False    53.8
True     46.2
Name: proportion, dtype: float64

Distribución en test:
IsToxic
False    54.0
True     46.0
Name: proportion, dtype: float64


# 7. Vectorización con TF-IDF

Convertimos el texto limpio en números que el modelo
pueda procesar.

Usamos TF-IDF (Term Frequency - Inverse Document Frequency),
una técnica que asigna un peso a cada palabra según
su importancia en el documento y en el dataset completo.

In [12]:
# --- Vectorización TF-IDF ---

# Creamos el vectorizador
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Entrenamos y transformamos train
X_train_tfidf = tfidf.fit_transform(X_train)

# Solo transformamos test
X_test_tfidf = tfidf.transform(X_test)

# Comprobamos dimensiones
print(f"X_train_tfidf: {X_train_tfidf.shape}")
print(f"X_test_tfidf:  {X_test_tfidf.shape}")

X_train_tfidf: (800, 2046)
X_test_tfidf:  (200, 2046)


### ✔ Verificación del Bloque 7 — Vectorización TF‑IDF

La vectorización se ha completado correctamente.

- El vectorizador TF‑IDF ha aprendido un vocabulario de **2046 features**  
  (palabras y bigramas relevantes para el modelo).
- La matriz de entrenamiento tiene forma **(800, 2046)**  
  y la de test **(200, 2046)**, lo que confirma que:
  - El vocabulario es consistente entre train y test.
  - No hay fuga de información (el `fit` se hizo solo con train).
- Las matrices se han guardado en formato `.npz`, optimizado para matrices dispersas.
- El vectorizador entrenado se ha guardado como `tfidf_vectorizer.pkl`  
  para poder reutilizarlo en el notebook de modelado.

**Conclusión:**  
El texto ya está completamente transformado en números y listo para entrenar el modelo en `03_modeling.ipynb`.  
Este bloque cierra el preprocesamiento de forma profesional.


El preprocesamiento del texto se ha completado correctamente y ha dejado el dataset listo para entrenar modelos de Machine Learning. Primero se realizó una limpieza del texto para eliminar ruido como URLs, emojis o signos innecesarios. Después, el texto se tokenizó, se eliminaron las stopwords y se aplicó lematización para reducir las palabras a su forma base y mejorar la generalización del modelo.

Todo el proceso se aplicó al dataset completo, generando la columna `Text_clean` con el texto preparado para vectorizar. Luego, los datos se dividieron en entrenamiento y prueba evitando fuga de información, y finalmente se utilizó TF-IDF para transformar el texto en una representación numérica lista para el modelado.

Con esto, el notebook deja preparado todo lo necesario para comenzar la fase de entrenamiento y evaluación de modelos en `03_modeling.ipynb`.
